
import os
from pathlib import Path
import shutil

root_dir = Path("/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05")  # Change to your root directory

for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
    for dirname in dirnames:
        if ".tif" in dirname:
            full_path = Path(dirpath) / dirname
            if not any(full_path.iterdir()):
                print(f"Removing empty directory: {full_path}")
                shutil.rmtree(full_path)
            else:
                print(f"Skipped non-empty directory: {full_path}")

# Stitching using ImageJ plugins
## First - we have to move files and folders according to their well positions 
We can also rename these files
Uses the helper functions from wells

also note the image dimenstions:
- Length:  205.0204 microns (2160)
- Width:  205.0204 microns (2160)
  - 10.5355 pixels/micron (1 px = 0.094917 microns)
- fields are roughly adjacent (~205.02) some tiny gaps ~0.001-0.003 microns

In [ ]:
from matplotlib import pyplot as plt
import cv2 as cv
import numpy as np
from make_file_scheme import *
from pathlib import Path
from PIL import Image
import numpy as np

input_folder = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05"
output_base = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05"

file_list = load_sorted_directory_list(input_folder)
nchannels = get_nchannels(file_list)
grouped_files_by_channel = group_files_by_channel(file_list, nchannels)

grouped_files_by_well = group_files_by_well(file_list, 40, nchannels)
#print_grouped_files(grouped_files_by_well)
block_a_files, block_b_files = get_well_stitching_block_paths(grouped_files_by_well)

In [ ]:
scale = 1/10.5355

x1 = 0.002460245 * np.pow(10, 6)  # Convert to micrometers
x2 = 0.002255224 * np.pow(10, 6) 
x3 = 0.002050204 * np.pow(10, 6) 
x4 = 0.001845183 * np.pow(10, 6) 
x5 = 0.001640163 * np.pow(10, 6) 

y1= 6.15061E-4 * np.pow(10, 6) 
y2 = 8.20082E-4 * np.pow(10, 6) 
y3 = 0.001025102 * np.pow(10, 6) 
y4 = 0.001230122 * np.pow(10, 6) 

print(f'x1-x2:{x1-x2},x2-x3:{x2-x3},x3-x4:{x3-x4},x4-x5:{x4-x5}')
print(f'y1-y2:{y1-y2},y2-y3:{y2-y3},y3-y4:{y3-y4}')
print(f'scale: {scale} microns/pixel, 1 pixel = {1/scale} microns, field of view: {scale*2160} or 205.0204 microns')

In [ ]:
#print_grouped_files(block_a_files)
test_wells = grouped_files_by_well[:36]
test_wells_a = block_a_files[:36]
print_grouped_files(block_a_files)
test_wells_b = block_b_files[:36]
test_well_ch2 = [file for file in test_wells if "ch2-" in str(file)]

output_folder_test = os.path.join(output_base,"test_stitching")
os.makedirs(output_folder_test, exist_ok=True)
#print_grouped_files(block_a_files)

print(len(test_wells))

In [ ]:
def extract_channel(filepath):
    """_summary_

    Args:
        filepath (Path object or String): a file path
    Returns:
        _type_: _description_
    """
    name = filepath
    if isinstance(filepath, Path):
        name = filepath.name
    query = re.search(r'ch(\d{1})', name)
    if query:
        ch_num = int(query.group(1))  
    else: 
        ch_num = None
    return ch_num
    
nchannels = 4

def rename_images_by_tiles_in_block(file_path, tile_index, block):
    """Run this within a loop to rename a file that is part of a group according to its image tile and the respective block it belongs to.
    Ideally run after a file is moved/copied to avoid unwanted changes

    Args:
        file_group (list): _description_
        file_path (Path or string): _description_
        block (string): _description_
    """    
    p = Path(file_path)
    # avoid out of bounds error
    ch_num = extract_channel(p)
    rowcol_location = get_plate_location(p, rowcolonly=True)
    new_name = f"MAX_ch{ch_num}-{rowcol_location}_block{block}_tile{tile_index:02d}{p.suffix}" #use fstring to define the new format
    new_path = p.with_name(new_name)
    print(new_path.name)
    print(f"Renaming {p.name} -> {new_name}")
    shutil.move(str(p), str(new_path))


In [ ]:
#testing the rename
for group in block_a_files:
    tile_index = 0 #define and reset tile index for each group
    block = "A"
    for idx, file_path in enumerate(group):
        #make the filename into a path object and get channel
        p = Path(file_path)
        ch_num = extract_channel(p)
        rowcol_location = get_plate_location(p, rowcolonly=True)
        # avoid out of bounds error
        if idx > 0:
            prev_file_ch_num = extract_channel(group[idx-1]) #get ch number from previous file in list 
            if ch_num < prev_file_ch_num and prev_file_ch_num == nchannels: 
                tile_index = tile_index +1 # only add to index when on "ch1" of the next image field and the previous img was the final channel
         
        new_name = f"MAX_ch{ch_num}-{rowcol_location}_block{block}_tile{tile_index:02d}{p.suffix}" #use fstring to define the new format
        new_path = p.with_name(new_name)
        print(new_path.name)
        print(f"Renaming {p.name} -> {new_name}")
        #shutil.move(str(p), str(new_path))
        #shutil.move(str(p), str(new_path))

In [ ]:
location = "r01c02f02"
print(location[:6])

In [ ]:
def copy_to_blocks(block_a_files, block_b_files, output_folder, use_input_folder = False, nchannels=4): 
    print(block_a_files[0][0])
    for block in [block_a_files,block_b_files]: #temporary; just a proof of concept for now
        for idx, group in enumerate(block):
            print(group[idx])
            #define and reset tile index for each group
            #condition when I'm testing out inputs and only working with one image set
            if isinstance(group,Path):
                filepath = group
                filename = filepath.name
                if use_input_folder:
                    output_folder = filepath.parent
                plate_location = get_plate_location(filepath, rowcolonly=True)
                location_path = os.path.join(output_folder, plate_location)
                #print(filename,str(parent_filepath),plate_location,location_path)
                
                if idx==0:
                    os.makedirs(location_path, exist_ok=True)
                    folder_a_path = os.path.join(location_path, "Block_A")
                    folder_b_path = os.path.join(location_path, "Block_B")
                    os.makedirs(folder_a_path, exist_ok=True)
                    os.makedirs(folder_b_path, exist_ok=True)
            else: #when looping through all the grouped files by well in the 2D list
                tile_index = 0 #reset tile index for each well
                root_path = group[0]

                if use_input_folder:
                    output_folder = root_path.parent
                plate_location = get_plate_location(root_path, rowcolonly=True)
                location_base_folder = os.path.join(output_folder, plate_location)
                os.makedirs(location_base_folder, exist_ok=True)
            
                folder_a_path = os.path.join(location_base_folder, "Block_A")
                folder_b_path = os.path.join(location_base_folder, "Block_B")
                #Create the new folders if they don't exist
                os.makedirs(folder_a_path, exist_ok=True)
                os.makedirs(folder_b_path, exist_ok=True)
    
            #debugging one well at a time
            if isinstance(group,Path):
                ch_num = extract_channel(block[idx])
                if idx > 0: #avoid out of bounds error
                        prev_file_ch_num = extract_channel(block[idx-1]) #get ch number from previous file in list 
                        if ch_num < prev_file_ch_num and prev_file_ch_num == nchannels: 
                            tile_index = tile_index +1 # only add to index when on "ch1" of the next image field and the previous img was the final channel      
                if filepath in block_a_files:
                    destination = shutil.copy2(filepath, folder_a_path)
                    print(f"Copied {filename} to {folder_a_path}")
                    rename_images_by_tiles_in_block(destination,tile_index)
                elif filepath in block_b_files:
                    destination = shutil.copy2(filepath, folder_b_path)
                    print(f"Copied {filename} to {folder_b_path}")
                    rename_images_by_tiles_in_block(destination,tile_index)
                    #increment the tile index when 
                
            else:   #loop thru all files in grouped by well
                for file_idx, file in enumerate(group):
                    if os.path.exists(file):
                        file = Path(file)
                        ch_num = extract_channel(file)
                        if file_idx > 0: #avoid out of bounds error
                            prev_file_ch_num = extract_channel(group[file_idx-1]) #get ch number from previous file in list 
                            if ch_num < prev_file_ch_num and prev_file_ch_num == nchannels: 
                                tile_index = tile_index +1 # only add to index when on "ch1" of the next image field and the previous img was the final channel      
                        if file in block_a_files[idx]:
                            destination = shutil.copy2(file, folder_a_path)
                            print(f"Copied {file.name} to {folder_a_path}")
                            rename_images_by_tiles_in_block(destination,tile_index, block = "A")
                        elif file in block_b_files[idx]:
                            destination = shutil.copy2(file, folder_b_path)
                            print(f"Copied {file.name} to {folder_b_path}")
                            rename_images_by_tiles_in_block(destination,tile_index, block = "B")
                        else:
                            print(f"{file.name} not in {root_path}")
                    else:
                        print(f"Warning: {file.name} not found in {file}")
        
        print("\nFile organization complete!")
    
#copy_to_blocks(test_wells, test_wells_a, test_wells_b, output_folder_test)

#copy_to_blocks(test_wells_a, test_wells_b, output_folder_test)

## Trying out PyimageJ
Note - can find plugin site [here](https://mvnrepository.com/artifact/sc.fiji)




In [ ]:
import imagej,scyjava
#ij = imagej.init('sc.fiji:fiji')
scyjava.config.add_options('-Xmx10g')
#j = imagej.init(['sc.fiji:fiji', "net.NIST-ISG-MIST"], add_legacy=True)
ij = imagej.init('/home/mattiazzilab/Fiji', mode='interactive', add_legacy=True) 
#NOTE can only have one instance running at a time, restart kernel if you see "2.xx/Inactive"


In [ ]:
print(f"ImageJ2 version: {ij.getVersion()}")
ij.getApp().getInfo(True)


In [ ]:
# Sanity checkcreate lists
python_list = [1, 2, 3, 4]
java_list = ij.py.to_java(python_list)

# modify one list
python_list[0] = 4

# check list contents
print(f"python_list: {python_list}\njava_list: {java_list}")

# Load an image.
image_url = 'https://imagej.net/images/clown.jpg'
jimage = ij.io().open(image_url)

# Convert the image from ImageJ2 to xarray, a package that adds
# labeled datasets to numpy (http://xarray.pydata.org/en/stable/).
image = ij.py.from_java(jimage)
ij.py.show(image, cmap='Blues')
# NB: This is not a built-in ImageJ command! It is the
# Plugins › Integral Image Filters › Mean command,
# which is part of mpicbg_, which is included with Fiji.


macro = """
#@ String name
#@ int age
#@ String city
#@output Object greeting
greeting = "Hello " + name + ". You are " + age + " years old, and live in " + city + "."
"""
args = {
    'name': 'Chuckles',
    'age': 13,
    'city': 'Nowhere'
}
result = ij.py.run_macro(macro, args)
print(result.getOutput('greeting'))

In [ ]:
#playing with ij ops
[op for op in ij.op().ops() if "gauss" in op]
print(ij.op().help("filter.gauss"))

def tubeness_1(image, sigma, calibration=[]):
    return ij.op().run("filter.tubeness", ij.py.jargs(image, sigma, calibration))

def gauss_1(img, sigmas):
    return ij.op().filter().gauss(img, sigmas)

web_image = ij.io().open('https://wsr.imagej.net/images/Cell_Colony.jpg')
ij.py.show(web_image)
ij.py.show(gauss_1(web_image, [2,0,2] ))
ij.py.show(tubeness_1(web_image, sigma=1))


In [ ]:
STITCHING_MACRO = """
#@ String grid_size_x
#@ String grid_size_y
#@ String tile_overlap
#@ String image_path
#@ String image_name
#@ String output_path
#@ String textfile_name
#@ String file_ext
 run("Grid/Collection stitching",
    "type=[Grid: snake by rows] order=[Right & Down                ]" +
    " grid_size_x=" + grid_size_x +
    " grid_size_y=" + grid_size_y +
    " tile_overlap=" + tile_overlap +
    " first_file_index_i=0"+
    " directory=" + image_path +
    " file_names=" + image_name + file_ext +
    " output_textfile_name=" + textfile_name +
    " fusion_method=[Linear Blending]"+
    " regression_threshold=0.30"+
    " max/avg_displacement_threshold=2.50"+
    " absolute_displacement_threshold=3.50"+
    " computation_parameters=[Save computation time (but use more RAM)]"+
    " image_output=[Write to disk]"+
    " output_directory="+output_path);
"""

images_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05/test_stitching/r02c02/Block_A"

def run_grid_stitching(images_path, ch_num=2):
    grid_size_x = 5
    grid_size_y = 4
    tile_overlap = 0

    image_path_folder = Path(images_path) 
    filenames = [f.name for f in image_path_folder.iterdir() if f.is_file()]
    location = get_plate_location(filenames[0], rowcolonly=True)
    block = images_path[-1]

    tile_variable = "{ii}"

    image_name = f"MAX_ch{ch_num}-{location}_block{block}_tile{tile_variable}"
    file_ext = ".tif"
    print(image_name)
    print(images_path)

    textfile_name = f"{location}_block{block}.txt"
    output_path = os.path.join(images_path, "grid_stitched_img")
    os.makedirs(output_path, exist_ok=True)

    args={"grid_size_x":grid_size_x,
                "grid_size_y":grid_size_y,
                "tile_overlap":tile_overlap,
                "image_path":images_path,
                "image_name":image_name,
                "file_ext":file_ext,
                "output_path":output_path,
                "textfile_name":textfile_name
                }
    _=ij.py.run_macro(STITCHING_MACRO, args)

run_grid_stitching(images_path)

In [ ]:
images_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05/test_stitching/r02c02/Block_A"
def run_mist(images_path, ch_num, reg_ch, output_path=None, image_name=None, from_metadata='false', display="true", headless="false"):
#imp = ij.getImage();
    plugin = "MIST"
    grid_size_x = 5
    grid_size_y = 4
    tile_overlap = 0
    blending_alpha = 1.0
    unitx = 0.09491718
    unity = 0.09491718
    

    image_path_folder = Path(images_path) 
    filenames = [f.name for f in image_path_folder.iterdir() if f.is_file()]
    location = get_plate_location(filenames[0], rowcolonly=True)
    block = images_path[-1]
    
    tile_variable = "{pp}"
    first_file_index_i = 0

    file_ext = ".tif"
    image_name = f"MAX_ch{ch_num}-{location}_block{block}_tile{tile_variable}{file_ext}"
    
    print(image_name)
    print(images_path)
    
    repeatability = 0 #0 by default
    overlap_uncertainty=5.0 #0-20

    planpath="/home/mattiazzilab/lib/fftw/fftPlans"
    fftwlibrarypath="/usr/local/lib"#"/usr/lib/x86_64-linux-gnu"
    fftw_filename = "libfftw3.so"

    if output_path is None:
        output_path = os.path.join(images_path, "MIST_stitched_img")
    
    os.makedirs(output_path, exist_ok=True)
    outfileprefix = f"ch-{ch_num}_{location}_block{block}_"

    if from_metadata.lower() == "true":
        textfile_name = os.path.join(output_path, f"ch-{reg_ch}_{location}_block{block}_global-positions-0.txt")
    else:
        textfile_name = "[]"
        #textfile_name = f"{location}_block{block}.txt"
    args = f"""
    gridwidth={grid_size_x} \
    gridheight={grid_size_y} \
    starttile={first_file_index_i} \
    imagedir={images_path} \
    filenamepattern={image_name} \
    filenamepatterntype=SEQUENTIAL \
    gridorigin=UL \
    assemblefrommetadata={from_metadata} \
    assemblenooverlap=false \
    globalpositionsfile={textfile_name} \
    numberingpattern=HORIZONTALCONTINUOUS \
    startrow=0 \
    startcol=0 \
    extentwidth={grid_size_x} \
    extentheight={grid_size_y} \
    timeslices=0 \
    istimeslicesenabled=false \
    outputpath={output_path} \
    displaystitching={display} \
    outputfullimage=true \
    outputmeta=true \
    outputimgpyramid=false \
    blendingmode=LINEAR \
    blendingalpha={blending_alpha} \
    compressionmode=LZW \
    outfileprefix={outfileprefix} \
    unit=MICROMETER \
    unitx={unitx} \
    unity={unity} \
    programtype=AUTO \
    numcputhreads=32 \
    loadfftwplan=true \
    savefftwplan=true \
    fftwplantype=MEASURE \
    fftwlibraryname=fftw3 \
    fftwlibraryfilename={fftw_filename} \
    planpath={planpath} \
    fftwlibrarypath={fftwlibrarypath} \
    stagerepeatability={repeatability}\
    horizontaloverlap={tile_overlap} \
    verticaloverlap={tile_overlap} \
    numfftpeaks=0 \
    overlapuncertainty={overlap_uncertainty} \
    isusedoubleprecision=true \
    isusebioformats=true \
    issuppressmodelwarningdialog=true \
    isenablecudaexceptions=false \
    translationrefinementmethod=EXHAUSTIVE \
    numtranslationrefinementstartpoints=16 \
    headless={headless} \
    loglevel=MANDATORY \
    debuglevel=HELPFUL
    """.replace('\n', ' ').strip()

    print(args)
    # ...pass `args` as a single string to Java or your plugin...
    '''
    args = f"gridwidth=5 gridheight=4 starttile={start_idx} imagedir={folder_path} filenamepattern={filenamepattern} filenamepatterntype=SEQUENTIAL gridorigin=UL assemblefrommetadata=false assemblenooverlap=false globalpositionsfile=[] numberingpattern=HORIZONTALCONTINUOUS startrow=0 startcol=0 extentwidth=5 extentheight=4 timeslices=0 istimeslicesenabled=false outputpath=/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/stitchtest_v2/block_b displaystitching=true outputfullimage=true outputmeta=true outputimgpyramid=false blendingmode=LINEAR blendingalpha=1.0 compressionmode=UNCOMPRESSED outfileprefix=testimg- unit=MICROMETER unitx=0.093 unity=0.093 programtype=AUTO numcputhreads=32 loadfftwplan=true savefftwplan=true fftwplantype=MEASURE fftwlibraryname=libfftw3 fftwlibraryfilename=libfftw3.dll planpath=/home/mattiazzilab/lib/fftw/fftPlans fftwlibrarypath=/home/mattiazzilab/lib/fftw stagerepeatability=0 horizontaloverlap=0.0 verticaloverlap=0.0 numfftpeaks=0 overlapuncertainty=NaN isusedoubleprecision=false isusebioformats=true issuppressmodelwarningdialog=false isenablecudaexceptions=false translationrefinementmethod=EXHAUSTIVE numtranslationrefinementstartpoints=16 headless=false loglevel=MANDATORY debuglevel=HELPFUL"
    args_dict = dict(item.split("=", 1) for item in args.split(" ") if "=" in item)
    print(args_dict["imagedir"])
    args_dict["imagedir"] = "/fake/path"
    print(args_dict["imagedir"])
    '''
    ij.py.run_plugin(plugin, args)

def multichannel_mist_stitching(images_path, nchannels=4, reg_ch=2, display="true", headless="false"):
    """Run MIST stitching for multiple channels in a given image path.

    Args:
        images_path (str): Path to the images.
        ch_num (int): Channel number to stitch.
        reg_ch (int): Reference channel number.
        from_metadata (str): Whether to use metadata for stitching.
        display (str): Whether to display the stitched image.
        headless (str): Whether to run in headless mode.
    """
        #run_mist(images_path, ch_num=ch_num, reg_ch=reg_ch, from_metadata=from_metadata, display=display, headless=headless)
    channels = list(range(1, nchannels + 1))  # Create a list of channel numbers
    channels.remove(reg_ch)  # Remove the registration channel from the list
    
    run_mist(images_path, ch_num=reg_ch, reg_ch=reg_ch, from_metadata='false', display=display, headless=headless)
    
    #now run the other channels
    for ch_num in channels:
        run_mist(images_path, ch_num=ch_num, reg_ch=reg_ch, from_metadata='true', display="false", headless="true")
        #run_mist(images_path, ch_num=i, reg_ch=4, from_metadata='true')

In [ ]:
test_parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328_rep05/test_stitching/"
for root, dirs, files in os.walk(test_parent_dir):
    for dir in dirs:
        if "Block" in dir:
            input_path = os.path.join(root, dir)
            registration_channel = 2
            multichannel_mist_stitching(input_path, nchannels=4, reg_ch=registration_channel, display="false", headless="true")